# V2 Phase 12 — Colab GPU pilot (6 × 3 = 18 cases)

**Before running:** Runtime → Change runtime type → **GPU** (T4 or better).

Runs a small, reproducible subset of the **frozen 140** through all three architectures independently:

1. `finqa_test_1000`, `1012`, `1017`, `1027`, `1039`, `1040`
2. Single-Agent, Multi-Agent, Multi-Agent + UQ
3. Raw JSONL + checkpoint/resume + duplicate prevention

Uses **llama_cpp + Qwen3-8B**. Threshold is `0.55` **smoke/demo — NOT LOCKED**.

Does **not** modify the frozen 140 or 40. Does **not** lock the threshold. Does **not** run the 420-case benchmark. Does **not** start Phase 13+.

## Setup

Push latest V2 (Phase 12 runner) to branch `cursor/empty-v2-workspace`, then run **this notebook on Colab GPU**.

Requires Phase 8 KB on Drive at `MyDrive/MSc-RAG/artifacts/knowledge_base/`.

**Outputs:** `results/raw/phase12_pilot/{run_id}/cases.jsonl`, checkpoint, `results/config/phase12_smoke_test.json`

If Colab disconnects: re-run section 5 with `--resume-latest`. Do not restart from question 1 by creating a new run.

## 1. Clone GitHub repo and enter V2

In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys

if platform.system() == 'Darwin' or not Path('/content').exists():
    raise RuntimeError('Open this notebook on Colab GPU. Do not run the Phase 12 pilot on the Mac.')

REPO_URL = 'https://github.com/syedsafiullah777/CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-.git'
BRANCH = 'cursor/empty-v2-workspace'
CLONE_DIR = Path('/content/capstone-rag')

if CLONE_DIR.exists():
    !rm -rf {CLONE_DIR}

print('Cloning branch:', BRANCH)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CLONE_DIR)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'git clone failed. Push V2/ to GitHub on branch {BRANCH!r} first.')

V2_ROOT = CLONE_DIR / 'V2'
if not (V2_ROOT / 'scripts' / 'run_pilot.py').is_file():
    raise FileNotFoundError(f'Phase 12 script missing at {V2_ROOT}. Push Phase 12 to GitHub first.')

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
os.environ['V2_REQUIRE_CUDA'] = '1'
os.environ['V2_FORBID_MOCK'] = '1'
print('OK — working in V2_ROOT:', V2_ROOT)
!git -C {CLONE_DIR} log -1 --oneline

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 3. Restore knowledge base from Drive (or rebuild)

Does **not** copy the Mac Chroma database. Reuses the Colab-built index from Phase 8 when available.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

V2 = Path('/content/capstone-rag/V2')
DRIVE_ROOT = Path('/content/drive/MyDrive/MSc-RAG')
restored = True

for rel in ('artifacts/knowledge_base/index', 'artifacts/knowledge_base/documents'):
    src = DRIVE_ROOT / rel
    dst = V2 / 'knowledge_base' / rel.split('/')[-1]
    if not src.is_dir():
        print('Missing on Drive:', src)
        restored = False
        break
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('restored', dst)

if not restored:
    print('Drive KB not found — falling back to build_index.py (Option B)...')
    !PYTHONPATH=. python scripts/build_index.py --distractors 50

## 4. Index preflight validation

In [ ]:
!PYTHONPATH=. python scripts/validate_kb_index.py

## 5. Phase 12 pilot (`llama_cpp`, 18 cases)

If this cell was interrupted, run the **resume** command in the next cell instead of this one.

In [ ]:
import os
os.environ['V2_REQUIRE_CUDA'] = '1'
os.environ['V2_FORBID_MOCK'] = '1'
!PYTHONPATH=. python scripts/run_pilot.py --backend llama_cpp --n-questions 6

## 5b. Resume after disconnect (only if section 5 did not finish)

Skips completed `{architecture}:{question_id}` keys. Retries failed cases. Does not restart from question 1.

In [ ]:
# Uncomment only after an interrupted run:
# import os
# os.environ['V2_REQUIRE_CUDA'] = '1'
# !PYTHONPATH=. python scripts/run_pilot.py --backend llama_cpp --n-questions 6 --resume-latest --retry-failed

## 6. Check raw results, schema, threshold NOT LOCKED

In [ ]:
import json
from pathlib import Path

from src.config import load_experiment_config

summary = Path('results/config/phase12_pilot_summary.json')
smoke = Path('results/config/phase12_smoke_test.json')
fp = Path('results/config/phase12_runtime_fingerprint.json')
print('summary:', summary.is_file(), 'smoke:', smoke.is_file(), 'fingerprint:', fp.is_file())

data = json.loads(summary.read_text())
print('status:', data.get('status'))
print('run_id:', data.get('run_id'))
print('backend:', data.get('backend'), 'device:', data.get('device'), 'gpu:', data.get('gpu'))
print('completed/failed/pending:', data.get('n_completed'), data.get('n_failed'), data.get('n_pending'))
print('threshold:', data.get('threshold'), 'locked:', data.get('threshold_locked'), data.get('threshold_note'))
print('question_ids:', data.get('question_ids'))

if data.get('device') == 'mps_capable_host':
    raise RuntimeError('This is a Mac result, not Colab T4.')
if data.get('backend') != 'llama_cpp':
    raise RuntimeError(f'Expected llama_cpp, got {data.get("backend")}')
if data.get('threshold_locked') is True:
    raise RuntimeError('Pilot must not lock the confidence threshold.')
if data.get('n_questions') != 6 or data.get('n_cases') != 18:
    raise RuntimeError('Pilot must be 6 questions × 3 architectures = 18 cases.')

raw = Path(data['raw_path'])
if not raw.is_file():
    raw = Path('/content/capstone-rag/V2') / data['raw_path']
cases = [json.loads(line) for line in raw.read_text().splitlines() if line.strip()]
keys = [c.get('case_key') for c in cases]
print('raw_cases:', len(cases), 'unique_keys:', len(set(keys)))
if len(set(keys)) != len(keys):
    raise RuntimeError('Duplicate case_key found in raw JSONL.')

fields = load_experiment_config().section('storage').get('raw_result_fields', [])
for case in cases:
    missing = [f for f in fields if f not in case]
    if missing:
        raise RuntimeError(f"{case.get('case_key')} missing {missing}")
    print(
        case.get('case_key'),
        'decision=', case.get('decision'),
        'n_evidence=', len(case.get('retrieved_evidence') or []),
        'confidence=', case.get('confidence'),
        'threshold=', case.get('threshold'),
        'latency=', case.get('latency_seconds'),
        'error=', case.get('error'),
    )
print('schema OK; threshold NOT LOCKED')

## 7. Save Phase 12 raw results and checkpoints to Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import shutil

drive.mount('/content/drive', force_remount=True)
V2 = Path('/content/capstone-rag/V2')
DRIVE = Path('/content/drive/MyDrive/MSc-RAG')

summary = json.loads((V2 / 'results' / 'config' / 'phase12_pilot_summary.json').read_text())
run_id = summary['run_id']

raw_src = V2 / 'results' / 'raw' / 'phase12_pilot' / run_id
raw_dest = DRIVE / 'results' / 'raw' / 'phase12_pilot' / run_id
raw_dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(raw_src, raw_dest, dirs_exist_ok=True)
print('copied raw', raw_dest)

ckpt_src = V2 / 'results' / 'checkpoints' / 'phase12_pilot'
ckpt_dest = DRIVE / 'checkpoints' / 'phase12_pilot'
if ckpt_src.is_dir():
    shutil.copytree(ckpt_src, ckpt_dest, dirs_exist_ok=True)
    print('copied checkpoints', ckpt_dest)

cfg_dest = DRIVE / 'configs' / 'phase12'
cfg_dest.mkdir(parents=True, exist_ok=True)
for name in (
    'phase12_runtime_fingerprint.json',
    'phase12_smoke_test.json',
    'phase12_pilot_summary.json',
):
    src = V2 / 'results' / 'config' / name
    if src.is_file():
        shutil.copy2(src, cfg_dest / name)
        print('copied', name)
print('Drive dest root:', raw_dest)